# Soup Lab Profile Offload28

Diagnostic-only profiles of immutable runtime06dc4b1 (all28 layers, slot-view cache).
Three full64-step child runs collect natural startup phase spans, CPU/CUDA timeline for steps4-6, and separate allocator history. All source/task/harness files are unchanged. No diagnostic wall time or memory peak is a qualification result.
The Python phase observer uses sys.monitoring on actual code objects; Dataset.map, fingerprint and formatter callables are never replaced. Timeline annotations and the diagnostic callback exist only in the timeline/memory processes.
Download stack-profile-raw.zip. This notebook neither runs paid GPUs nor alters the old competition notebook.


In [ ]:
import subprocess, sys
from pathlib import Path
TARGET = '06dc4b1957fbe1e23b4b32fa420d3205890fa553'
DRIVER_COMMIT = '49f333512ab4d49abe24023c8230e73d1f41de59'
REPOSITORY = 'https://github.com/kok-o/soup-kernel-lab.git'
DRIVER = Path('/kaggle/working/profile-driver')
OUT = Path('/kaggle/working/stack-profile')
def run(args, **kwargs):
    return subprocess.run(args, check=True, **kwargs)
assert not DRIVER.exists() and not OUT.exists(), 'Use a fresh session'
run(['git', 'init', str(DRIVER)])
run(['git', '-C', str(DRIVER), 'config', 'core.autocrlf', 'false'])
run(['git', '-C', str(DRIVER), 'config', 'core.eol', 'lf'])
run(['git', '-C', str(DRIVER), 'remote', 'add', 'origin', REPOSITORY])
run(['git', '-C', str(DRIVER), 'fetch', '--depth', '1', 'origin', DRIVER_COMMIT])
run(['git', '-C', str(DRIVER), 'checkout', '--detach', 'FETCH_HEAD'])
assert subprocess.check_output(['git','-C',str(DRIVER),'rev-parse','HEAD'], text=True).strip() == DRIVER_COMMIT
print(subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], text=True))
run([sys.executable, 'research/test_phase_monitor.py'], cwd=DRIVER)


In [ ]:
# Installation is outside the official timed child processes.
run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao', 'xformers', 'flash-attn', 'liger-kernel', 'unsloth', 'unsloth-zoo'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.11.0', '--index-url', 'https://download.pytorch.org/whl/cu128'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'triton==3.6.0', 'soup-cli[train]==0.77.0', 'transformers==5.16.1', 'peft==0.20.0', 'trl==0.29.1', 'bitsandbytes==0.50.2', 'accelerate==1.14.0', 'datasets==4.8.5', 'pytest'])


In [ ]:
run([sys.executable, 'research/run_stack_profile.py', '--commit', TARGET, '--out', str(OUT)], cwd=DRIVER)


In [ ]:
import json
r=json.loads((OUT/'PROFILE_PROVENANCE.json').read_text())
print('DIAGNOSTIC ONLY:',r['commit'])
print(json.dumps({k:v['exit_code'] for k,v in r['results'].items()},indent=2))
